# Autograd & Manual Gradient Updates in PyTorch

## What you will get from this notebook

By the end you will understand:

1. **How PyTorch autograd works** — how `.backward()` computes gradients automatically through the computation graph
2. **Two ways to train a model**:
   - Using `nn.Module` + `torch.optim` (the standard high-level API)
   - Updating parameters manually with raw tensors (the low-level way)
3. **Leaf tensors vs non-leaf tensors** — why `a -= lr * a.grad` is correct but `a = a - lr * a.grad` silently breaks training
4. **Why `torch.no_grad()` is required** during manual parameter updates
5. **What `optimizer.step()` and `optimizer.zero_grad()` actually do** under the hood

---

**Task:** Recover the coefficients of `y = 4x³ + 8x + 7` from data using gradient descent.

## Learn a regression

In [ ]:
import torch
import torch.nn as nn

import matplotlib.pyplot as plt

In [ ]:
# Define the targe model
def target(x):
    y = 4* x**3 + x*8 + 7
    return y

-4073


In [ ]:
class LinearModel(nn.Module):
    def __init__(self):
        super(LinearModel, self).__init__()
        self.a = nn.Parameter(torch.randn(1))
        self.c = nn.Parameter(torch.randn(1))
        self.d = nn.Parameter(torch.randn(1))
    
    def forward(self, x):
        x_3 = x**3
        return x_3*self.a + x*self.c +self.d

x = torch.randn((10,10))
print(x.shape)
y = target(x)
print(y.shape, y.mean())
model = LinearModel()
y_pre = model(x)
print(y_pre.shape,y_pre.mean())

torch.Size([10, 10])
torch.Size([10, 10]) tensor(6.9496)
torch.Size([10, 10]) tensor(-0.0568, grad_fn=<MeanBackward0>)


In [ ]:
# training
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

for _ in range(5000):
    optimizer.zero_grad()
    x = torch.randn((10,10))
    y = target(x)
    y_pre = model(x)
    loss = criterion(y, y_pre)
    
    loss.backward()
    optimizer.step()
    
    if _ % 100 ==0:
        print("step: ", _, "loss: ", loss)

step:  0 loss:  tensor(393.7271, grad_fn=<MseLossBackward0>)
step:  100 loss:  tensor(263.8571, grad_fn=<MseLossBackward0>)
step:  200 loss:  tensor(499.3244, grad_fn=<MseLossBackward0>)
step:  300 loss:  tensor(161.0765, grad_fn=<MseLossBackward0>)
step:  400 loss:  tensor(185.0490, grad_fn=<MseLossBackward0>)
step:  500 loss:  tensor(109.2031, grad_fn=<MseLossBackward0>)
step:  600 loss:  tensor(43.8387, grad_fn=<MseLossBackward0>)
step:  700 loss:  tensor(22.9708, grad_fn=<MseLossBackward0>)
step:  800 loss:  tensor(19.7183, grad_fn=<MseLossBackward0>)
step:  900 loss:  tensor(10.2800, grad_fn=<MseLossBackward0>)
step:  1000 loss:  tensor(5.6572, grad_fn=<MseLossBackward0>)
step:  1100 loss:  tensor(3.6920, grad_fn=<MseLossBackward0>)
step:  1200 loss:  tensor(3.4887, grad_fn=<MseLossBackward0>)
step:  1300 loss:  tensor(2.4321, grad_fn=<MseLossBackward0>)
step:  1400 loss:  tensor(2.0392, grad_fn=<MseLossBackward0>)
step:  1500 loss:  tensor(4.7326, grad_fn=<MseLossBackward0>)
step

In [57]:
print(model.a, model.c, model.d)

Parameter containing:
tensor([4.0026], requires_grad=True) Parameter containing:
tensor([7.9870], requires_grad=True) Parameter containing:
tensor([7.0000], requires_grad=True)


## Manual Gradient Update (Without Optimizer)

Instead of using `torch.optim`, we can update parameters manually.  
This is a useful exercise to understand exactly what an optimizer does internally.

---

### ⚠️ Key Subtlety: `a -= ...` vs `a = a - ...`

These look almost identical but behave very differently in PyTorch:

| Expression | What happens | Effect on `a` |
|---|---|---|
| `a -= lr * a.grad` | **In-place** update (`a.sub_()`) | `a` stays a **leaf tensor** — `.grad` still works next step |
| `a = a - lr * a.grad` | Creates a **new tensor**, rebinds name `a` | `a` becomes **non-leaf** — `.grad` is `None` next step ❌ |

### Why does it matter?

PyTorch only stores gradients in `.grad` for **leaf tensors** — tensors created directly by the user with `requires_grad=True`.  
When you write `a = a - ...`, you replace `a` with a computed tensor, which is **not** a leaf.

```python
a = torch.ones(1, requires_grad=True)

# ❌ Out-of-place: a becomes non-leaf
a = a - lr * a.grad    # a.is_leaf → False, a.grad will be None

# ✅ In-place: a stays a leaf tensor
with torch.no_grad():
    a -= lr * a.grad   # a.is_leaf → True, a.grad still works
```

### Why `torch.no_grad()`?

Without it, the in-place update itself gets tracked by autograd — modifying a tensor that is part of the computation graph. Wrapping in `torch.no_grad()` ensures the update is **not tracked** and doesn't corrupt the graph.

In [ ]:
# Training without an optimizer — manual gradient updates
criterion = nn.MSELoss()
lr = 0.01

a = torch.ones(1, requires_grad=True)
b = torch.ones(1, requires_grad=True)
c = torch.ones(1, requires_grad=True)

for _ in range(5000):
    x = torch.randn((10, 10))
    y = target(x)
    y_pre = a * x**3 + b * x + c
    loss = criterion(y, y_pre)

    loss.backward()

    with torch.no_grad():   # wrap update: prevents autograd from tracking the in-place op
        a -= lr * a.grad    # ✅ in-place: a stays a leaf tensor
        b -= lr * b.grad
        c -= lr * c.grad

        # Manually zero gradients — equivalent to optimizer.zero_grad()
        a.grad = None
        b.grad = None
        c.grad = None

    if _ % 100 == 0:
        print("step:", _, "  loss:", loss.item())

step:  0 loss:  tensor(415.7595, grad_fn=<MseLossBackward0>) None None None
step:  100 loss:  tensor(3.3649, grad_fn=<MseLossBackward0>) None None None
step:  200 loss:  tensor(0.8865, grad_fn=<MseLossBackward0>) None None None
step:  300 loss:  tensor(0.1601, grad_fn=<MseLossBackward0>) None None None
step:  400 loss:  tensor(0.0384, grad_fn=<MseLossBackward0>) None None None
step:  500 loss:  tensor(0.0094, grad_fn=<MseLossBackward0>) None None None
step:  600 loss:  tensor(0.0026, grad_fn=<MseLossBackward0>) None None None
step:  700 loss:  tensor(0.0004, grad_fn=<MseLossBackward0>) None None None
step:  800 loss:  tensor(5.8349e-05, grad_fn=<MseLossBackward0>) None None None
step:  900 loss:  tensor(1.7553e-05, grad_fn=<MseLossBackward0>) None None None
step:  1000 loss:  tensor(3.4206e-06, grad_fn=<MseLossBackward0>) None None None
step:  1100 loss:  tensor(6.2558e-07, grad_fn=<MseLossBackward0>) None None None
step:  1200 loss:  tensor(2.0191e-07, grad_fn=<MseLossBackward0>) None

In [67]:
print(a, b, c)

tensor([4.0000], requires_grad=True) tensor([8.0000], requires_grad=True) tensor([7.0000], requires_grad=True)


In [ ]:
# Demonstrate: -= (in-place) keeps leaf, = creates a non-leaf
import torch

a_test = torch.ones(1, requires_grad=True)
print("Initial is_leaf:", a_test.is_leaf)           # True ✅

# Out-of-place: reassigns a_test to a new computed tensor
a_out = a_test - 1
print("After a = a - 1, is_leaf:", a_out.is_leaf)   # False ❌ — .grad will NOT accumulate

# In-place: modifies the same tensor object
a_inplace = torch.ones(1, requires_grad=True)
with torch.no_grad():
    a_inplace -= 1
print("After a -= 1,    is_leaf:", a_inplace.is_leaf)  # True ✅ — .grad still works

### Summary: The correct manual update pattern

```python
with torch.no_grad():
    param -= lr * param.grad   # ✅ in-place, stays leaf
    param.grad = None          # ✅ zero grad for next step
```

This is exactly what `optimizer.step()` + `optimizer.zero_grad()` do under the hood for SGD.  
`nn.Parameter` objects are leaf tensors by design, which is why `model.parameters()` + an optimizer always works correctly.